### This is a Universal test file. Every function we change or make, needs to be tested here. Also the utility of each file can be found here

#### Test for llms.py file
* You will need ollama to run opensource llms. Download here : https://ollama.com/
* install llama3.2 to test the examples
* Test to check that simple openai messages and chatollama messages works
* Test to check structured format works with both platforms
* Test to check tool calling is working with openai

In [ ]:
# All imports here

from utils.llms import *
from pydantic import BaseModel,Field
from utils.keys import set_api_keys
set_api_keys()

Openai key set successfully


In [7]:
# Test 1 : Is simple messaging working

# messages we are assuming openai style. Will be auto updated to langchain style. we will try both with only user and system
messages_test_1 = [{"role":"user", "content":"Hello How are you"}]
messages_test_2 = [{"role":"system","content":"Respond in victorian english"},{"role":"user", "content":"Hello How are you"}]

# responses from openai
response_openai_non_thinking = run_llm("gpt-4o-mini",messages_test_1,schema=None,tools=None)
response_openai_thinking = run_llm("o1-mini",messages_test_1,schema=None,tools=None)
response_openai_non_thinking_2 = run_llm("gpt-4o",messages_test_2,schema=None,tools=None)
response_openai_thinking_2 = run_llm("gpt-4.1-mini",messages_test_2,schema=None,tools=None)

print("Openai Runs completed")

# responses from opensource here llama2
response_open_source_non_thinking = run_llm("llama3.2",messages_test_1,schema=None,tools=None)
response_open_source_non_thinking_2 = run_llm("llama3.2",messages_test_2,schema=None,tools=None)

print("Llama2 runs completed")

print("Response from openai non thinking 1", response_openai_non_thinking)
print("Response from openai non thinking 2", response_openai_non_thinking_2)
print("Response from openai thinking 1", response_openai_thinking)
print("Response from openai thinking 2", response_openai_thinking_2)
print("Response from open source non thinking 1", response_open_source_non_thinking)
print("Response from open source non thinking 2", response_open_source_non_thinking_2)
   

Openai Runs completed
Llama2 runs completed
Response from openai non thinking 1 Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?
Response from openai non thinking 2 Good day to you! I find myself in fine spirits, thank you kindly for inquiring. And how might you be faring on this pleasant day?
Response from openai thinking 1 Hello! I'm doing well, thank you for asking. How can I assist you today?
Response from openai thinking 2 Good morrow to thee! I trust this missive findeth thee in fine fettle. How doth thy day fare?
Response from open source non thinking 1 I'm just a language model, so I don't have emotions or feelings like humans do. However, I'm functioning properly and ready to assist you with any questions or tasks you may have! How can I help you today?
Response from open source non thinking 2 Dear compatriot, I daresay I am in a state of optimal felicity, thank you for inquiring after my well-being. The gentl

In [8]:
# Test 2 : Is structured schema working. I will make a test schema to test it. 

class TestSchema(BaseModel):
    bengali : str = Field("Translate to bengali") # add common languages to test further
    hindi : str = Field("Translate to hindi")

messages = [{"role":"system", "content":"given the text translate it to bengali and hindi"},{"role":"user", "content":"Peter piper picked a bunch of pickled peppers"}]

# responses from openai
response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=TestSchema,tools=None)
response_openai_thinking = run_llm("o1-mini",messages,schema=TestSchema,tools=None)

print("Openai Runs completed")

# responses from opensource here llama2
response_open_source_non_thinking = run_llm("llama3.2",messages,schema=TestSchema,tools=None)

print("Llama2 runs completed")

print("Response from openai non thinking", response_openai_non_thinking)
print("Response from openai thinking", response_openai_thinking)
print("Response from open source non thinking", response_open_source_non_thinking)


c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\llms.py:118: UserWarning: NOTE : Openai reasoning models do not support structured output so will  provide the structure in prompt and try. Susceptible to failure
  warnings.warn("NOTE : Openai reasoning models do not support structured output so will  provide the structure in prompt and try. Susceptible to failure",UserWarning)


Openai Runs completed
Llama2 runs completed
Response from openai non thinking bengali='পিটার পাইপার একটি ঝুড়ি আচার করা মরিচ তুলেছিল।' hindi='पीटर पाइपर ने अचार वाले मिर्चों का एक गुच्छा उठाया।'
Response from openai thinking bengali='পিটার পাইপার অনেকগুলো আচার করা মরিচ সংগ্রহ করেছিলেন' hindi='पीटर पाइपर ने अचार किए हुए मिर्चों का एक गुच्छा चुना'
Response from open source non thinking bengali='পিটার পাইপার একটি বন্ধু গজের মশলা আঁচড়ে আঁচড়ে পেপার' hindi='पीटर पाइपर ने एक बूंद की मिश्रित सब्जियों का संग्रह किया'


In [9]:
# Test 3 tool use only openai models supported here, While llama models can call tools, using it for NGD is not feasible untill we can accomodate slightly higher model sizes
# Here I will make a custom tool. Get the inputs from the LLM and then run the tool and given the outputs again to demonstrate a simple use

# simple substraction tool
def substract_two_numbers(a:float, b:float):
    return a-b

# simple addition tool
def add_two_numbers(a:float,b:float):
    return a+b

# Here I will make the tool description. For OS NGD will need to make a single function and describe the tool as below
tool_definition = [
    {
        "type":"function",
        "name":"substract_two_numbers",
        "description":"Simple tool to substract two numbers. Takes a and b as input calculates a-b",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"first number input"
                },
                "b":{
                    "type":"number",
                    "description":"second number input"
                }
            }
        }
    },

    {
        "type":"function",
        "name":"add_two_numbers",
        "description":"Simple tool to add two numbers. Takes a and b as input calculates a+b",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"first number input"
                },
                "b":{
                    "type":"number",
                    "description":"second number input"
                }
            }
        }
    },]

# Now we make the messages

messages = [{"role":"system","content":"you have 2 tools one to add numbers and another to substract. Use it to solve user equation. Multiple tool calls can be required to solve 1 equation"},
            {"role":"user","content":"solve: (8.95-6.71)"},
            {"role":"user","content":"solve: (8.95+6.71)"},]

response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=None,tools=tool_definition)

# I would have written a custom execute tool code but will keep it pending now. Sometimes tool calls are supposed to be dependent on each other so I will write the code here and we can transfer to a function later
# We can multithread the above code no problem at all but maybe later
attempts = 0
while(response_openai_non_thinking.output[-1].type=="function_call" and attempts < 4):
    fn_calls = [i for i in response_openai_non_thinking.output if i.type=="function_call"]

    for call in fn_calls:
        messages.append(call)
        args = json.loads(call.arguments)

        if call.name == "substract_two_numbers":
            result = substract_two_numbers(args["a"],args["b"])
            messages.append({"type":"function_call_output", "call_id":call.call_id,"output":str(result)})
        
        if call.name == "add_two_numbers":
            result = add_two_numbers(args["a"],args["b"])
            messages.append({"type":"function_call_output", "call_id":call.call_id,"output":str(result)})
    response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=None,tools=tool_definition)
    # No infinite loops for llms
    attempts = attempts + 1

print(response_openai_non_thinking.output_text)

The results are:

- \( 8.95 - 6.71 = 2.24 \)
- \( 8.95 + 6.71 = 15.66 \)


In [10]:
# Sometimes we may want to execute tools sequentially as an example ((6.45-2.34)+5.67) so first substract then add


messages = [{"role":"system","content":"you have 2 tools one to add numbers and another to substract. Use it to solve user equation. Multiple tool calls can be required to solve 1 equation"},
            {"role":"user","content":"solve: ((6.45-2.34)+5.67)"}]

response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=None,tools=tool_definition,additional_args={"temperature":0,"max_tokens":2048,"parallel_tool_calls":False})

attempts = 0
while(response_openai_non_thinking.output[-1].type=="function_call" and attempts < 4):
    fn_calls = [i for i in response_openai_non_thinking.output if i.type=="function_call"]
    # I am printing this to verify sequential execution is this is printed twice the 2 tools are called not in parallel but sequentially
    print(fn_calls)

    for call in fn_calls:
        messages.append(call)
        args = json.loads(call.arguments)

        if call.name == "substract_two_numbers":
            result = substract_two_numbers(args["a"],args["b"])
            messages.append({"type":"function_call_output", "call_id":call.call_id,"output":str(result)})
        
        if call.name == "add_two_numbers":
            result = add_two_numbers(args["a"],args["b"])
            messages.append({"type":"function_call_output", "call_id":call.call_id,"output":str(result)})
    response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=None,tools=tool_definition)
    # No infinite loops for llms
    attempts = attempts + 1

print(response_openai_non_thinking.output_text)

[ResponseFunctionToolCall(arguments='{"a":6.45,"b":2.34}', call_id='call_Ub02l3yaN3UIyuzPEYhSJqb4', name='substract_two_numbers', type='function_call', id='fc_05fcce5257eb8efe0068f267cc55bc8196ae39e953171763bc', status='completed')]
[ResponseFunctionToolCall(arguments='{"a":4.11,"b":5.67}', call_id='call_WAmDfaupA0gOxvWOGzOAGCg1', name='add_two_numbers', type='function_call', id='fc_05fcce5257eb8efe0068f267cf03188196bb6ff9348a248561', status='completed')]
The solution to the equation \(((6.45 - 2.34) + 5.67)\) is approximately \(9.78\).
